# MLB win-probability model

The sport-agnostic Elo+feature pipeline on **38k real MLB games** (2010–2026, `data/mlb.duckdb`), plus two starting-pitcher features: a quick **run-prevention proxy** and a real **FIP** rating (K/BB/HR from per-start pitcher game logs — isolates the pitcher from his team). Out-of-sample throughout.

In [ ]:
import sys
from pathlib import Path
import duckdb, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import os as _os
if _os.getenv('NB_DARK'):
    sns.set_theme(style='darkgrid', rc={
        'figure.facecolor':'#0d1117','axes.facecolor':'#161b22','savefig.facecolor':'#0d1117',
        'axes.edgecolor':'#30363d','axes.labelcolor':'#c9d1d9','text.color':'#c9d1d9',
        'axes.titlecolor':'#c9d1d9','xtick.color':'#8b949e','ytick.color':'#8b949e',
        'grid.color':'#21262d'})
else:
    sns.set_theme(style='whitegrid')
ROOT = Path.cwd()
while not (ROOT/'src'/'sportsball').exists() and ROOT != ROOT.parent: ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src')); sys.path.insert(0, str(ROOT/'scripts')); sys.path.insert(0, str(ROOT/'research'/'mlb'))
from sportsball.pipelines._elo import walk_forward
from train_eval_duckdb import _pipeline
from pitcher_features import pitcher_run_prevention, rolling_fip, fip_advantage
from sklearn.metrics import log_loss, brier_score_loss, accuracy_score
K, HFA = 4, 24

In [ ]:
con = duckdb.connect(str(ROOT/'data'/'mlb.duckdb'), read_only=True)
rows = con.execute('''SELECT game_date,home_team,away_team,home_score,away_score,
    home_sp_id,away_sp_id,game_pk FROM games ORDER BY game_date, game_pk''').fetchall()
logs = con.execute('SELECT pitcher_id,game_pk,game_date,outs,so,bb,hr,hbp FROM pitcher_logs').fetchall()
con.close()
results = [(r[0],r[1],r[2],r[3],r[4]) for r in rows]
pgames  = [(r[5],r[6],r[3],r[4]) for r in rows]        # (h_sp,a_sp,hs,as) for the proxy
fgames  = [(r[5],r[6],r[7]) for r in rows]             # (h_sp,a_sp,game_pk) for FIP
frows, snaps = walk_forward(results, K, HFA, mov_enabled=True, carry=0.75, gap_days=120, form_window=15)
X = np.array([r.features for r in frows]); y = np.array([1 if r.actual>=1 else 0 for r in frows])
elo_p = np.array([r.exp_home for r in frows])
proxy = np.array(pitcher_run_prevention(pgames, window=10)['pitcher_adv_home'])
fip_map, league = rolling_fip(logs, window=15, min_starts=3)
fip = np.array(fip_advantage(fgames, fip_map, league))
cut = int(0.85*len(X))
covered = np.mean([1 if (r[5],r[7]) in fip_map and fip_map[(r[5],r[7])] is not None else 0 for r in rows])
print(f'{len(X)} games | test {len(y)-cut} | base rate {y[cut:].mean():.3f} | home starts with a FIP rating: {covered:.0%}')

## Does a real (FIP) pitcher rating help where the proxy didn't?

The proxy (opponent runs) was within noise because it folds in bullpen/defense. FIP isolates the pitcher (K, BB, HR). Stack them up out-of-sample.

In [ ]:
def rep(name, p):
    return {'model':name,'accuracy':round(accuracy_score(y[cut:],(p[cut:]>=0.5).astype(int)),4),
            'log_loss':round(log_loss(y[cut:],p[cut:],labels=[0,1]),4),'brier':round(brier_score_loss(y[cut:],p[cut:]),4)}
def fit_p(M):
    m=_pipeline().fit(M[:cut], y[:cut]); out=np.empty(len(y))
    out[:cut]=m.predict_proba(M[:cut])[:,1]; out[cut:]=m.predict_proba(M[cut:])[:,1]; return out
home_const = np.full(len(y), y[:cut].mean())
models = [('always-home (base)', home_const), ('Elo only', elo_p),
          ('9-feature', fit_p(X)),
          ('9-feat + proxy', fit_p(np.column_stack([X, proxy]))),
          ('9-feat + FIP', fit_p(np.column_stack([X, fip]))),
          ('9-feat + proxy + FIP', fit_p(np.column_stack([X, proxy, fip]))),
          ('FIP advantage only', fit_p(fip.reshape(-1,1)))]
tbl = pd.DataFrame([rep(n,p) for n,p in models]); tbl

In [ ]:
base_ll = tbl.loc[2,'log_loss']
for i,lbl in [(3,'proxy'),(4,'FIP'),(5,'proxy+FIP')]:
    print(f'{lbl:10} lift vs 9-feature: log_loss {tbl.loc[i,"log_loss"]-base_ll:+.4f}, acc {tbl.loc[i,"accuracy"]-tbl.loc[2,"accuracy"]:+.4f}')

## Best starters by FIP (sanity)

Career FIP-core in this dataset (≥80 starts, lower=better) — should surface real aces.

In [ ]:
agg={}
for (pid,gpk,date,outs,so,bb,hr,hbp) in logs:
    a=agg.setdefault(pid,[0,0,0,0,0]); a[0]+=hr; a[1]+=bb; a[2]+=hbp; a[3]+=so; a[4]+=outs
names=dict(); 
import duckdb as _d
con=_d.connect(str(ROOT/'data'/'mlb.duckdb'), read_only=True)
for pid,nm in con.execute('SELECT DISTINCT home_sp_id,home_sp FROM games WHERE home_sp_id IS NOT NULL').fetchall(): names[pid]=nm
con.close()
recs=[]
for pid,(hr,bb,hbp,so,outs) in agg.items():
    if outs/3>=80*5:  # ~>=80 starts worth of innings
        recs.append((names.get(pid,str(pid)), (13*hr+3*(bb+hbp)-2*so)/(outs/3), outs/3))
fipdf=pd.DataFrame(recs, columns=['pitcher','fip_core','innings']).sort_values('fip_core').head(12)
plt.figure(figsize=(9,6)); sns.barplot(data=fipdf, y='pitcher', x='fip_core', color='seagreen')
plt.xlabel('FIP-core (lower = better)'); plt.title('Best starters by FIP-core (career, this dataset)'); plt.tight_layout(); plt.show()

## Calibration (best model) & skill over time

In [ ]:
best_p = fit_p(np.column_stack([X, fip]))
from sklearn.calibration import calibration_curve
fp, mp = calibration_curve(y[cut:], best_p[cut:], n_bins=8, strategy='quantile')
fig,ax=plt.subplots(1,2,figsize=(13,5))
ax[0].plot([0,1],[0,1],'--',color='gray'); ax[0].plot(mp,fp,'o-',color='seagreen'); ax[0].set_title('Calibration (9-feat + FIP)'); ax[0].set_xlabel('predicted'); ax[0].set_ylabel('observed')
win=400; correct=(best_p[cut:]>=0.5).astype(int)==y[cut:]
ax[1].plot(pd.Series(correct).rolling(win,min_periods=50).mean().values, color='seagreen', label='model')
ax[1].plot(pd.Series(y[cut:]==1).rolling(win,min_periods=50).mean().values, color='gray', ls='--', label='always-home')
ax[1].axhline(0.5,color='crimson',lw=.6); ax[1].legend(); ax[1].set_title(f'Skill over time (roll {win})')
plt.tight_layout(); plt.show()

## Verdict

**The honest, slightly counterintuitive answer: pregame starting-pitcher identity barely moves MLB win probability beyond team strength.** Even a *properly isolated* FIP rating (K/BB/HR, 94% coverage) adds only ~-0.0006 log-loss on top of team-Elo — within noise, no better than the crude run-prevention proxy. FIP *alone* (0.542) barely beats always-home and loses to team-Elo.

Why: (1) team-Elo already encodes rotation quality; (2) a single start is enormously variable — even an ace loses ~35% of starts, so a 15-start FIP is a noisy read on one game; (3) opponent offense, bullpen, and pure variance dominate. Baseball is simply that random at the single-game level — which is the real reason the model tops out near 56%. Levers that might still help are situational (park, lineup, rest) or the market itself (`market_logit` from ingested MLB closing odds), not pitcher identity.